# Walk-Forward Modeling for Financial Time Series
### A Realistic Evaluation Framework for Trading Models

**Author:** Lucas Minchillo  
**Goal:** Demonstrate proper walk-forward validation for financial ML models  
**Focus:** Avoiding overfitting and data leakage in sequential markets

## Executive Summary

Financial time series present unique challenges for machine learning:
- Low signal-to-noise ratio
- Non-stationarity (regime changes)
- Reflexivity and market adaptation

Traditional ML validation techniques (random splits, k-fold CV) are invalid in this domain.

In this notebook, we:
- Implement a **walk-forward training and evaluation framework**
- Use **logistic regression** as a baseline directional model
- Evaluate models using **expected log return**, not MSE or accuracy
- Compare **expanding vs rolling training windows**
- Show how statistical edge translates into trading performance

This notebook serves as a foundation for more complex models (tree-based, deep learning).

## Problem Definition

**Objective:** We aim to predict the **direction of the next-period log return** of Bitcoin using
lagged log returns as features.

**Framing:** 
- Binary classification problem (up/down)
- **Primary metric:** Profitability (expected log return)
- **Secondary metric:** Directional accuracy



## Why Standard ML Metrics Are Insufficient

Metrics such as:
- Accuracy
- MSE / MAE
- AUC

do **not** tell us whether a model is profitable.

Instead, we evaluate:
- **Expected value (EV)** of trade log returns
- Directional accuracy as a secondary diagnostic


## Walk-Forward Modeling


>"Walk-Forward
>Analysis is an evaluation of a trading strategy exclusively on the basis >of
>its performance on out-of-sample price data—data that have not been seen
>by the optimization process" (Robert Pardo, 2008)

Unlike static train-test splits, walk-forward modeling:
- Respects temporal ordering
- Continuously incorporates new information
- Avoids look-ahead bias
- Simulates real trading deployment


In our case, we are not testing a trading strategy, altghouth we are testing the evaluation of models that could be used as basis for a trading strategy. 


| Component | Specification |
|-----------|---------------|
| Asset | Bitcoin (BTC-USD) |
| Timeframe | 1-hour candles |
| Period | 2019–2026 |
| Target | Next-period log return direction |
| Features | Lagged log returns (AR-style) |
| Baseline Model | Logistic Regression |
| Training Schemes | Expanding window, Rolling 30-day window |
| Re-training Frequency | Weekly |
| AR Order Tested | 1–7 lags |


## Reproducibility & Hardware

- Fixed random seeds across NumPy, PyTorch, Python
- GPU acceleration when available
- Deterministic training for fair model comparison


### Environment Setup

In [ ]:
import os
import random
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from torch import nn, optim

from pathlib import Path
import kagglehub
warnings.filterwarnings("ignore")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = 99) -> np.random.Generator:
    """Set the seed for reproducibility across Python, NumPy, and PyTorch."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)

    # Modern NumPy Generator (Ruff NPY002)
    # We return the generator so it can be used throughout the script (Ruff F841)
    rng = np.random.default_rng(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    return rng


# Capture the generator to avoid unused variable warnings
rng = set_seed()

In [ ]:
# Initial dataset download
downloaded_path = kagglehub.dataset_download(
    "novandraanugrah/bitcoin-historical-datasets-2018-2024",
)

def load_bitcoin_data(dir_path: str | Path = downloaded_path, tf: str = "1h") -> pd.DataFrame:
    """Load Bitcoin historical data from a CSV file into a pandas DataFrame.

    Args:
        dir_path: The directory containing the dataset files.
        tf: The time frame of the data (e.g., '1h', '1d').

    """
    filename = f"btc_{tf}_data_2018_to_2025.csv"
    file_path = os.path.join(dir_path, filename)
    print(file_path)
    return pd.read_csv(file_path, parse_dates=["Close time"])

In [ ]:
df = load_bitcoin_data(downloaded_path, tf="1h")


In [ ]:

# Convert types
df["Open time"] = pd.to_datetime(df["Open time"])
df["Close time"] = pd.to_datetime(df["Open time"])

# 3. Avoid inplace=True (PD002) and filter
df = df.drop(columns=["Ignore"])
df = df.loc[df["Close time"] > datetime(2019, 12, 20)]
df = df.drop_duplicates()

## Feature Engineering

- Log returns
- Lagged log returns (AR features)
- Binary direction target


"Autoregressive models are heavily used in economic forecasting. An autoregressive model relates a time
series variable to its past values. This section discusses the basic ideas of autoregressions models, shows
how they are estimated and discusses an application to forecasting GDP growth using R." FROM ITER 

$Y_{t}=\beta_{0}+\beta_{1}Y_{t-1}+u_{l}$

In [ ]:
df["close_log_return"] = np.log(df["Close"] / df["Close"].shift(1))
for lag in range(1, 11):
    df[f"close_log_return_lag_{lag}"] = df["close_log_return"].shift(lag)
df["close_log_return_dir"] = df["close_log_return"].map(lambda x: 1 if x > 0 else 0)
df = df.dropna().set_index("Close time")


## Model: Logistic Regression

We use logistic regression as a **baseline linear classifier**.

Reasons:
- Interpretable
- Fast to retrain
- Common benchmark in financial ML


$p(X)={\frac{e^{\beta_{0}+\beta_{1}X_{1}+\cdots+\beta_{p}X_{p}}}{1+e^{\beta_{0}+\beta_{1}X_{1}+\cdots+\beta_{p}X_{p}}}}.$

In [ ]:
# -------------------------------------------------------
# MODEL DEFINITION (CUDA OPTIMIZED)
# -------------------------------------------------------
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_size):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_size, 1)

    def forward(self, x):
        return self.linear(x)

In [ ]:
def train_regression_model(X_train, y_train, n_epochs=5000, lr=0.01):
    """Highly optimized training loop for CUDA."""
    set_seed(99)
    n_features = X_train.shape[1]

    model = LogisticRegressionModel(n_features).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    x_tensor = torch.as_tensor(X_train, dtype=torch.float32, device=device)
    y_tensor = torch.as_tensor(y_train, dtype=torch.float32, device=device).unsqueeze(1)

    model.train()
    for epoch in range(n_epochs):
        optimizer.zero_grad(set_to_none=True)  # set_to_none=True is faster
        outputs = model(x_tensor)
        loss = criterion(outputs, y_tensor)
        loss.backward()
        optimizer.step()

    return model

## Walk-Forward Training & Evaluation Engine

This function:
- Generates re-training dates
- Trains models using rolling or expanding windows
- Evaluates strictly out-of-sample data
- Collects trade-level returns


In [ ]:
# -------------------------------------------------------
# WALK-FORWARD LOGIC (UNCHANGED)
# -------------------------------------------------------
def get_recalculation_dates(df, frequency="Q"):
    if isinstance(frequency, int):
        dates = pd.date_range(
            start=df.index.min(), end=df.index.max(), freq=f"{frequency}H",
        )
    else:
        dates = df.resample(frequency).mean().index
    return [date for date in dates if date < df.index.max()]

In [ ]:
def evaluate_logistic_profitability(model, X_test, y_test_returns, y_test_direction):
    model.eval()
    x_tensor = torch.as_tensor(X_test, dtype=torch.float32, device=device)

    with torch.no_grad():
        y_pred_logits = model(x_tensor)
        y_pred_proba = torch.sigmoid(y_pred_logits)
        y_pred_binary = (y_pred_proba >= 0.5).float()

    predictions = y_pred_binary.cpu().squeeze().numpy()
    signals = np.where(predictions == 1, 1, -1)
    trade_returns = signals * y_test_returns

    ev = trade_returns.mean()
    accuracy = np.mean(predictions == y_test_direction)
    buy_rate = np.mean(predictions == 1)

    return ev, accuracy, buy_rate, signals, trade_returns

In [ ]:
def walk_forward_train_predict(
    df,
    features,
    target_direction,
    target_return,
    window_type="rolling",
    window_size="90D",
    retrain_freq="Q",
    initial_train_size=None,
):
    recalc_dates = get_recalculation_dates(df, retrain_freq)
    all_predictions, model_metrics, models_trained = [], [], {}

    for i, recalc_date in enumerate(recalc_dates):
        if window_type == "expanding":
            train_mask = df.index <= recalc_date
            if initial_train_size:
                start_date = recalc_date - pd.Timedelta(initial_train_size)
                train_mask = train_mask & (df.index >= start_date)
        else:
            start_date = recalc_date - (
                pd.Timedelta(window_size)
                if isinstance(window_size, str)
                else timedelta(hours=window_size)
            )
            train_mask = (df.index > start_date) & (df.index <= recalc_date)

        pred_end = recalc_dates[i + 1] if i < len(recalc_dates) - 1 else df.index.max()
        pred_mask = (df.index > recalc_date) & (df.index <= pred_end)

        if not df[train_mask].shape[0] or not df[pred_mask].shape[0]:
            continue

        # Training
        train_df = df[train_mask]
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(train_df[features].values)
        y_train = train_df[target_direction].values

        model = train_regression_model(X_train_scaled, y_train)

        # Testing
        pred_df = df[pred_mask]
        X_pred_scaled = scaler.transform(pred_df[features].values)
        ev, acc, br, signals, t_ret = evaluate_logistic_profitability(
            model,
            X_pred_scaled,
            pred_df[target_return].values,
            pred_df[target_direction].values,
        )

        all_predictions.append(
            pd.DataFrame(
                {
                    "date": pred_df.index,
                    "actual_return": pred_df[target_return],
                    "actual_direction": pred_df[target_direction],
                    "predicted_direction": signals > 0,
                    "signal": signals,
                    "trade_return": t_ret,
                    "model_date": recalc_date,
                    "period": i,
                },
            ),
        )

        model_metrics.append(
            {
                "model_date": recalc_date,
                "ev": ev,
                "accuracy": acc,
                "train_samples": len(X_train_scaled),
            },
        )

    return pd.concat(all_predictions), pd.DataFrame(model_metrics), models_trained


In [ ]:

def compare_walk_forward_strategies(df, ar_order=7):
    features_dict = {
        f"AR{i}": [f"close_log_return_lag_{j}" for j in range(1, i + 1)]
        for i in range(1, ar_order + 1)
    }
    strategies = [
        {
            "name": "Expanding_Window_Weekly",
            "window_type": "expanding",
            "retrain_freq": "W",
            "initial_train_size": "30D",
        },
        {
            "name": "Rolling_30D_Weekly",
            "window_type": "rolling",
            "window_size": "30D",
            "retrain_freq": "W",
        },
    ]
    results = {}
    for strategy in strategies:
        strategy_results = []
        for model_name, features in features_dict.items():
            preds, metrics, _ = walk_forward_train_predict(
                df,
                features,
                "close_log_return_dir",
                "close_log_return",
                **{k: v for k, v in strategy.items() if k != "name"},
            )
            if not metrics.empty:
                strategy_results.append(
                    {
                        "Model": model_name,
                        "Lags": len(features),
                        "Mean_EV": metrics["ev"].mean(),
                        "Mean_Accuracy": metrics["accuracy"].mean(),
                        "Num_Periods": len(metrics),
                    },
                )
        results[strategy["name"]] = pd.DataFrame(strategy_results).sort_values(
            "Mean_EV", ascending=False,
        )
    return results




In [ ]:
# Final Run
results = compare_walk_forward_strategies(df, ar_order=7)
for name, res_df in results.items():
    print(f"\nSTRATEGY: {name}\n", res_df)

## Results

We report:
- Mean Expected Value (EV) per model
- Mean directional accuracy
- Number of walk-forward periods

A positive EV indicates statistical edge, even if accuracy is only slightly above 50%.


### 🚀 Using device: `cuda`

---

### STRATEGY: **Expanding Window (Weekly)**

| Model | Lags | Mean EV | Mean Accuracy | Num Periods |
|:-----:|:----:|--------:|--------------:|------------:|
| AR3 | 3 | 0.000072 | 0.529772 | 316 |
| AR5 | 5 | 0.000063 | 0.525067 | 316 |
| AR2 | 2 | 0.000059 | 0.529905 | 316 |
| AR4 | 4 | 0.000056 | 0.527269 | 316 |
| AR7 | 7 | 0.000051 | 0.523312 | 316 |
| AR1 | 1 | 0.000048 | 0.523921 | 316 |
| AR6 | 6 | 0.000045 | 0.524980 | 316 |

---

### STRATEGY: **Rolling 30D (Weekly)**

| Model | Lags | Mean EV | Mean Accuracy | Num Periods |
|:-----:|:----:|--------:|--------------:|------------:|
| AR3 | 3 | 0.000072 | 0.529991 | 316 |
| AR5 | 5 | 0.000061 | 0.524887 | 316 |
| AR2 | 2 | 0.000058 | 0.529417 | 316 |
| AR4 | 4 | 0.000055 | 0.527392 | 316 |
| AR7 | 7 | 0.000055 | 0.523900 | 316 |
| AR1 | 1 | 0.000050 | 0.523909 | 316 |
| AR6 | 6 | 0.000048 | 0.524867 | 316 |


## Interpretation

Key observations:
- AR(3) consistently outperforms higher-order models
- More complexity does not imply better performance
- Rolling and expanding windows yield similar EVs
- Accuracy ~52–53% is sufficient for profitability

This highlights the importance of proper evaluation over model complexity.


## Limitations & Future Work

- No transaction costs included
- Single asset only
- Linear model baseline
- Focus on Walk Forward
- 

Planned extensions of this notebooks:
- Feature Engineering
- Feature Selection
- Model evalution
- Trading Strategy Creation
- Tree-based models (Random Forest, XGBoost)
- Volatility-scaled position sizing





## Conclusion


This notebook provides a robust framework for walk-forward analysis of financial time series models. The key takeaways:

1. Temporal integrity is crucial – always use walk-forward or similar time-aware validation

2. Simplicity often wins – AR(3) outperformed more complex models

3. Focus on profitability metrics – EV and Sharpe ratio are more important than accuracy

4.  Regular retraining helps adapt to changing market conditions

The framework is extensible and can serve as a foundation for more sophisticated trading systems.

## References

- Gareth, J., Witten, D., Hastie, T., & Tibshirani, R. (2021). An Introduction to Statistical Learning: with Applications in R (2nd ed.). Springer.

- Hanck, C., Arnold, M., Gerber, A., & Schmelzer, M. (2024). Introduction to Econometrics with R. https://www.econometrics-with-r.org

- Pardo, R. (2008). The Evaluation and Optimization of Trading Strategies (2nd ed.). John Wiley & Sons.

*Note: This notebook is for educational purposes only. Past performance does not guarantee future results. Cryptocurrency trading involves substantial risk.*